In [22]:
from influxdb_client import InfluxDBClient, Point
from influxdb_client.client.write_api import SYNCHRONOUS
import csv
from datetime import datetime

In [27]:
url = "http://tradehunter.duckdns.org:8086"
token = "I7MLtkx-A_vJ3-JITkcYQqmhtxvc3zABaMBD-gmWY1eP2rcy4BqMzH_sVhNC7LhyDrGJKdIOxHptmgkuy28VFA=="
org = "TradeHunter"
bucket = "Trade Hunter Real Time Data"

In [28]:
#Initialize the InfluxDB client
client = InfluxDBClient(url=url, token=token, org=org)
#Initialize the synchronous write API
write_api = client.write_api(write_options=SYNCHRONOUS)

In [29]:
# Function to import data from a CSV file to InfluxDB
def import_csv_to_influx(file_path, measurement):
    with open(file_path, 'r') as csv_file:
        csv_reader = csv.DictReader(csv_file)

        for row in csv_reader:

            # Convert the date string to a timestamp
            timestamp = int(datetime.strptime(row['Date'], '%Y-%m-%d').timestamp()) * 1000000000

            # Create an InfluxDB data point for each row
            data = Point(measurement).time(timestamp)

            # Iterate over each column in the row (excluding 'Date')
            for key, value in row.items():
                if key != 'Date':
                    # Convert non-date values to floats and add them as fields
                    data.field(key, float(value))

            # Write the data point to InfluxDB
            write_api.write(bucket=bucket, record=data, timeout=20)

In [30]:
import_csv_to_influx('IBEX-1994-2020.csv', 'Internal_Factors')

In [ ]:
# Close the connection when finished
client.close()